<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading 

**Chapter 08 &mdash; CFD Trading with Oanda**

&copy; Dr Yves J Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:training@tpq.io) | [dyjh](http://twitter.com/dyjh)

<img src="https://hilpisch.com/pyalgo_cover_color.png" width="40%">

## The Oanda API

`pip install --upgrade git+https://github.com/yhilpisch/tpqoa.git`

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
import tpqoa

In [ ]:
api = tpqoa.tpqoa('../../../pyalgo.cfg')  # adjust path

## Retrieving Historical Data

### Looking Up Instruments Available for Trading

In [ ]:
api.get_instruments()

### Backtesting a Momentum Strategy on Minute Bars

In [ ]:
help(api.get_history)

In [ ]:
instrument = 'EUR_USD'
start = '2020-05-27'
end = '2020-05-29'
granularity = 'M1'
price = 'M'

In [ ]:
data = api.get_history(instrument, start, end,
                      granularity, price)

In [ ]:
data.info()

In [ ]:
data[['c', 'volume']].head()

In [ ]:
import numpy as np

In [ ]:
data['returns'] = np.log(data['c'] / data['c'].shift(1))

In [ ]:
cols = []

In [ ]:
for momentum in [15, 30, 60, 120]:
    col = 'position_{}'.format(momentum)
    data[col] = np.sign(data['returns'].rolling(momentum).mean())
    cols.append(col)

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
import matplotlib as mpl
mpl.rcParams['font.family'] = 'serif'

In [ ]:
strats = ['returns']

In [ ]:
for col in cols:
    strat = 'strategy_{}'.format(col.split('_')[1])
    data[strat] = data[col].shift(1) * data['returns']
    strats.append(strat)

In [ ]:
data[strats].dropna().sum().apply(np.exp)

In [ ]:
data[strats].dropna().cumsum(
    ).apply(np.exp).plot(figsize=(10, 6));

### Factoring In Leverage and Margin

In [ ]:
data[strats].dropna().cumsum().apply(
            lambda x: x * 20).apply(np.exp).plot(figsize=(10, 6));

## Working with Streaming Data

In [ ]:
instrument = 'EUR_USD'

In [ ]:
api.stream_data(instrument, stop=10)

## Placing Orders

In [ ]:
help(api.create_order)

In [ ]:
api.create_order(instrument, 1000)

In [ ]:
api.create_order(instrument, -1500)

In [ ]:
api.create_order(instrument, 500)

## Implementing Trading Strategies in Real-Time

In [ ]:
# api.stream_data??

In [ ]:
# api.on_success??

In [ ]:
import MomentumTrader as MT

In [ ]:
mt = MT.MomentumTrader('../../../pyalgo.cfg',
                       instrument=instrument,
                       bar_length='5s',
                       momentum=6,
                       units=10000)

In [ ]:
mt.stream_data(mt.instrument, stop=150)

In [ ]:
oo = mt.create_order(instrument, units=-mt.position * mt.units,
                     ret=True, suppress=True)
oo

### Retrieving Account Information

In [ ]:
api.get_account_summary()

In [ ]:
api.get_transactions(tid=int(oo['id']) - 2)

In [ ]:
api.print_transactions(tid=int(oo['id']) - 18)

## Regression Trader (Idea)

In [ ]:
p = np.arange(5) + np.random.standard_normal(5) * 2

In [ ]:
p

In [ ]:
x = np.arange(len(p))

In [ ]:
reg = np.polyfit(x, p, deg=1)

In [ ]:
x_ = np.arange(len(x) + 1)

In [ ]:
pred = np.polyval(reg, x_)

In [ ]:
plt.plot(x, p, 'ro')
plt.plot(x_, pred);

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>